# Fase 6 · Informe de la rueda de prensa (Bloque 2)

**Objetivo:** generar, para cada rueda de la demo, el informe que muestra la app y que lee la voz sintética: tono frente al histórico, momentos clave y un briefing de unas 150 palabras con citas a la transcripción. Es la versión solo texto: los segundos de cada cita y las señales de voz y cara se incorporan cuando estén disponibles.

In [1]:
# autoreload solo vigila nuestro código (src), no las librerías instaladas
%load_ext autoreload
%autoreload 1
%aimport src.config, src.text.rag, src.text.report, src.text.report_eval

import json

import pandas as pd

from src.config import ROOT, event_dir
from src.text.models import llm
from src.text.report import cargar_historico, preparar_evento, redactar_informe, guardar_informe
from src.text.report_eval import revisar

resultados_dir = ROOT / "benchmarks" / "results" / "informe"
resultados_dir.mkdir(parents=True, exist_ok=True)

## 1. Diseño del informe

- **Tono frente al histórico.** La etiqueta (dovish, slightly dovish, neutral, slightly hawkish, hawkish) corresponde al quintil del percentil de la rueda entre las ruedas de la era Lagarde. Es una lectura relativa: la puntuación bruta depende de la calibración del modelo, mientras que la posición frente a las demás ruedas es comparable y es lo que interesa a una mesa ("más hawkish que el 80 % de las ruedas"). Los percentiles se calculan solo con las ruedas hasta la fecha del evento: el informe dice lo que habría dicho ese día, sin información posterior.
- **Declaración frente a respuestas.** Cada parte tiene su propio percentil: una diferencia entre el texto acordado por el Consejo de Gobierno y las respuestas espontáneas es una señal relevante para el mercado.
- **Momentos clave.** Las dos frases relevantes más hawkish y las dos más dovish que no repiten literalmente la rueda anterior, como mucho una por párrafo. Entre una reunión y la siguiente se copia en torno a una cuarta parte de la declaración ("We are not pre-committing to a particular rate path"), y esas frases no aportan información nueva. En la fase 5 podrán reordenarse con el tono de voz y la expresión facial.
- **Briefing, en dos partes.** (1) Un párrafo generado por código con la decisión de tipos (y su tamaño, tal como lo anuncia la declaración), la etiqueta de tono y los percentiles: las cifras de nuestro modelo no pasan por el LLM, así que no las puede alterar. (2) Cinco frases del LLM, en tercera persona y una por línea: tres mensajes clave y dos riesgos, cada una con la cita del fragmento que la respalda. El LLM recibe la transcripción completa en fragmentos numerados, con cada pregunta de los periodistas como contexto no citable. El código separa las frases (también cuando el LLM las escribe seguidas en un párrafo, con el mismo segmentador que el corpus), las agrupa en dos párrafos y renumera las citas, que tienen el mismo formato que las del chat.
- **Por qué este formato.** Una primera versión pedía al LLM el informe completo: tres párrafos libres a partir de las métricas y la transcripción. En las doce pruebas (dos modelos, seis ruedas), once informes salieron sin ninguna cita, la extensión media superó el objetivo (213 palabras con Qwen3 y 285 con Llama, frente a 130-180) y el primer párrafo se limitaba a enumerar percentiles. Con unas 6.000 palabras de contexto, ninguno de los dos modelos mantuvo las instrucciones de formato. Frases sueltas, cada una con su cita y con la instrucción repetida al final del mensaje, son más fáciles de cumplir y de comprobar.
- **Cumplimiento.** Las mismas reglas que en el chat: solo información de la rueda, sin previsiones, opiniones ni recomendaciones de inversión, y un aviso fijo en el informe ("For information purposes only. This is not investment advice.").
- **Voz.** `briefing_tts` es el mismo texto sin marcadores de cita y con las abreviaturas desarrolladas ("%" → "per cent"), para la voz sintética.

## 2. Candidatos y ruedas de prueba

| Candidato | Motivo |
|---|---|
| Qwen3 8B (MLX, 4 bits) | El LLM de la postura y del chat: si basta, la app mantiene un único modelo en memoria. |
| Llama 3.1 8B Instruct (MLX, 4 bits) | Alternativa de tamaño equivalente, ya evaluada en la fase 2. |

Ruedas de prueba: las seis más recientes (febrero-septiembre de 2026), candidatas a la demo. Incluyen cuatro mantenimientos y las dos subidas de 2026, antes y después del shock energético causado por el conflicto en Oriente Medio.

## 3. Comprobaciones automáticas y regla de decisión

- `cifras_sin_respaldo`: cifras de las frases del LLM que no aparecen en la transcripción que recibió (las del párrafo de tono salen del código). Es la comprobación principal: una cifra inventada es el error más grave en un informe para una mesa de tesorería.
- `frases` y `frases_con_cita`: frases del LLM (se piden cinco) y cuántas citan al menos un fragmento existente.
- `recomendaciones`: expresiones de recomendación ("recommend", "investors should", "overweight"...). `citas`: fragmentos distintos citados. `palabras`: extensión del briefing completo. `segundos`: tiempo de generación en un Mac M4 con 24 GB.
- **Regla, fijada antes de ver los resultados.** Gana el modelo con menos cifras sin respaldo en el total de las seis ruedas; si empatan, el de mayor proporción de frases con cita; después, el que use menos lenguaje de recomendación; si persiste el empate, Qwen3, que ya está en memoria para la postura y el chat. Una ejecución anterior de este formato mostró por qué se mide la proporción y no el número: a veces los modelos escriben las cinco frases seguidas en un párrafo y, contadas por líneas, ese informe sumaba una sola frase, de modo que el recuento premiaba al modelo que más veces separaba las frases en líneas, no al que más citaba.
- **Criterio de aceptación.** El ganador debe tener, en cada uno de los seis informes, cero cifras sin respaldo, cero expresiones de recomendación y al menos cuatro de sus cinco frases con cita. Si no lo cumple, el formato se revisa de nuevo.
- **Límite.** Las comprobaciones detectan cifras inventadas, pero no cifras reales mal atribuidas (un dato de salarios presentado como inflación), ni miden la calidad de la redacción o si cada cita respalda su frase. Por eso se revisan además, a mano, los dos informes de la rueda de referencia (sección 5).

In [2]:
corpus, frases, indice = cargar_historico()

CANDIDATOS = ["mlx-community/Qwen3-8B-4bit", "mlx-community/Meta-Llama-3.1-8B-Instruct-4bit"]
RUEDAS = sorted(indice["date"])[-6:]
REFERENCIA = "2026-09-10"

eventos = {fecha: preparar_evento(fecha, corpus, frases, indice) for fecha in RUEDAS}
print(f"Ruedas: {', '.join(RUEDAS)}")
print(f"Mensaje al LLM de la rueda de referencia: {len(eventos[REFERENCIA]['mensaje'].split())} palabras, "
      f"{len(eventos[REFERENCIA]['fragmentos'])} fragmentos citables")

Ruedas: 2026-02-05, 2026-03-19, 2026-04-30, 2026-06-11, 2026-07-23, 2026-09-10
Mensaje al LLM de la rueda de referencia: 6117 palabras, 77 fragmentos citables


In [3]:
informes, filas = {}, []
for modelo in CANDIDATOS:
    for fecha, evento in eventos.items():
        informe = redactar_informe(evento, modelo)
        informes[(modelo, fecha)] = informe
        filas.append({"modelo": modelo, "fecha": fecha, **revisar(informe, evento["mensaje"])})
        print(f"{modelo.split('/')[-1]} | {fecha}: {filas[-1]['segundos']} s", flush=True)
    llm.liberar()   # descarga el modelo antes de cargar el siguiente

comprobaciones = pd.DataFrame(filas)
comprobaciones.to_csv(resultados_dir / "comprobaciones.csv", index=False)
# Informes completos, para revisarlos sin volver a generarlos
with open(resultados_dir / "informes.json", "w", encoding="utf-8") as f:
    json.dump([{"modelo": m, "fecha": fe, **i} for (m, fe), i in informes.items()], f, indent=2, ensure_ascii=False)

/Users/eduardogonzalezarroyo/Developer/hawk-and-dove_BCE/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 1786.42it/s]


Qwen3-8B-4bit | 2026-02-05: 44.3 s
Qwen3-8B-4bit | 2026-03-19: 42.7 s
Qwen3-8B-4bit | 2026-04-30: 53.0 s
Qwen3-8B-4bit | 2026-06-11: 57.9 s
Qwen3-8B-4bit | 2026-07-23: 53.5 s
Qwen3-8B-4bit | 2026-09-10: 66.4 s


Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 2099.25it/s]


Meta-Llama-3.1-8B-Instruct-4bit | 2026-02-05: 52.6 s
Meta-Llama-3.1-8B-Instruct-4bit | 2026-03-19: 58.2 s
Meta-Llama-3.1-8B-Instruct-4bit | 2026-04-30: 70.8 s
Meta-Llama-3.1-8B-Instruct-4bit | 2026-06-11: 66.7 s
Meta-Llama-3.1-8B-Instruct-4bit | 2026-07-23: 60.2 s
Meta-Llama-3.1-8B-Instruct-4bit | 2026-09-10: 70.9 s


## 4. Resultados

In [4]:
resumen = comprobaciones.groupby("modelo").agg(
    cifras_sin_respaldo=("cifras_sin_respaldo", "sum"), frases_con_cita=("frases_con_cita", "sum"),
    frases=("frases", "sum"), recomendaciones=("recomendaciones", "sum"),
    palabras_media=("palabras", "mean"), segundos_media=("segundos", "mean"))
resumen.insert(3, "prop_con_cita", resumen["frases_con_cita"] / resumen["frases"])

# Regla: menos cifras sin respaldo; mayor proporción de frases con cita; menos recomendaciones; Qwen3
orden = resumen.assign(sin_cita=-resumen["prop_con_cita"], no_es_qwen=[m != CANDIDATOS[0] for m in resumen.index])
ganador = orden.sort_values(["cifras_sin_respaldo", "sin_cita", "recomendaciones", "no_es_qwen"]).index[0]

# Criterio de aceptación, informe a informe
del_ganador = comprobaciones[comprobaciones["modelo"] == ganador]
aceptado = ((del_ganador["cifras_sin_respaldo"] == 0) & (del_ganador["recomendaciones"] == 0)
            & (del_ganador["frases_con_cita"] >= 4)).all()
print(f"Ganador según la regla: {ganador} | criterio de aceptación: {'cumplido' if aceptado else 'NO cumplido'}")
resumen.round(2)

Ganador según la regla: mlx-community/Qwen3-8B-4bit | criterio de aceptación: cumplido


,cifras_sin_respaldo,frases_con_cita,frases,prop_con_cita,recomendaciones,palabras_media,segundos_media
modelo,,,,,,,
mlx-community/Meta-Llama-3.1-8B-Instruct-4bit,0,30,30,1.0,0,211.0,63.23
mlx-community/Qwen3-8B-4bit,0,30,30,1.0,0,166.5,52.97


In [5]:
# Detalle por rueda, con las cifras sin respaldo para revisarlas
comprobaciones.set_index(["modelo", "fecha"])

palabras  frases  \
modelo                                        fecha                          
mlx-community/Qwen3-8B-4bit                   2026-02-05       175       5   
                                              2026-03-19       166       5   
                                              2026-04-30       148       5   
                                              2026-06-11       188       5   
                                              2026-07-23       153       5   
                                              2026-09-10       169       5   
mlx-community/Meta-Llama-3.1-8B-Instruct-4bit 2026-02-05       213       5   
                                              2026-03-19       245       5   
                                              2026-04-30       222       5   
                                              2026-06-11       197       5   
                                              2026-07-23       200       5   
                                              2026-09-10       189       5   

                                                          frases_con_cita  \
modelo                                        fecha                         
mlx-community/Qwen3-8B-4bit                   2026-02-05                5   
                                              2026-03-19                5   
                                              2026-04-30                5   
                                              2026-06-11                5   
                                              2026-07-23                5   
                                              2026-09-10                5   
mlx-community/Meta-Llama-3.1-8B-Instruct-4bit 2026-02-05                5   
                                              2026-03-19                5   
                                              2026-04-30                5   
                                              2026-06-11                5   
                                              2026-07-23                5   
                                              2026-09-10                5   

                                                          citas  \
modelo                                        fecha               
mlx-community/Qwen3-8B-4bit                   2026-02-05      5   
                                              2026-03-19      5   
                                              2026-04-30      5   
                                              2026-06-11      4   
                                              2026-07-23      5   
                                              2026-09-10      4   
mlx-community/Meta-Llama-3.1-8B-Instruct-4bit 2026-02-05      4   
                                              2026-03-19      5   
                                              2026-04-30      5   
                                              2026-06-11      5   
                                              2026-07-23      5   
                                              2026-09-10      4   

                                                          cifras_sin_respaldo  \
modelo                                        fecha                             
mlx-community/Qwen3-8B-4bit                   2026-02-05                    0   
                                              2026-03-19                    0   
                                              2026-04-30                    0   
                                              2026-06-11                    0   
                                              2026-07-23                    0   
                                              2026-09-10                    0   
mlx-community/Meta-Llama-3.1-8B-Instruct-4bit 2026-02-05                    0   
                                              2026-03-19                    0   
                                              2026-04-30                    0   
                                              2026-06-11

## 5. Revisión de la rueda de referencia

Los dos informes del 10 de septiembre de 2026, con sus citas, para revisar a mano la redacción y si cada cita respalda lo que se afirma.

In [6]:
for modelo in CANDIDATOS:
    informe = informes[(modelo, REFERENCIA)]
    print(f"=== {modelo} ===\n\n{informe['briefing']}\n")
    for i, c in enumerate(informe["citations"], start=1):
        print(f"[{i}] {c['source']}\n    {c['snippet'][:250]}...")
    print()

=== mlx-community/Qwen3-8B-4bit ===

The ECB raised its key interest rates by 25 basis points. The Hawk & Dove stance index rates the press conference as slightly hawkish: it ranks at the 69th percentile of the 55 press conferences held since December 2019, up from the 56th percentile at the previous meeting on 23 July 2026. The monetary policy statement ranks at the 69th percentile and the answers in the Q&A at the 64th.

The ECB expects inflation to remain above target through 2027, with headline inflation averaging 3.0% in 2026 and gradually declining thereafter [1]. The ECB anticipates economic growth to be resilient, with a baseline projection of 0.9% for 2026 and upward revisions for subsequent years [1]. The ECB maintains a data-dependent approach, emphasizing the need to monitor inflation risks and economic developments closely [2].

The ECB highlights the Middle East conflict and energy price volatility as key upward risks to inflation [3]. The ECB identifies geopolitical tens

## 6. Informes de la demo

Se guardan `stance.csv` y `summary.json` de las seis ruedas con el modelo ganador (los informes ya generados: la decodificación es determinista). Debajo, los seis briefings y los momentos clave de la rueda de referencia, para revisarlos tal como los verá el usuario.

In [7]:
for fecha, evento in eventos.items():
    guardar_informe(evento, informes[(ganador, fecha)], ganador)
print(f"Guardados en data/events/ ({len(eventos)} ruedas) con {ganador}")

for fecha in RUEDAS:
    resumen_rueda = json.loads((event_dir(fecha) / "summary.json").read_text(encoding="utf-8"))
    print(f"=== {fecha} ===\n\n{resumen_rueda['briefing']}\n")

print("=== Momentos clave de la rueda de referencia ===")
for momento in json.loads((event_dir(REFERENCIA) / "summary.json").read_text(encoding="utf-8"))["key_moments"]:
    print(f"- {momento['reason']} ({momento['score']:+.2f}): {momento['text']}")

Guardados en data/events/ (6 ruedas) con mlx-community/Qwen3-8B-4bit
=== 2026-02-05 ===

The ECB kept its key interest rates unchanged. The Hawk & Dove stance index rates the press conference as neutral: it ranks at the 42nd percentile of the 50 press conferences held since December 2019, down from the 59th percentile at the previous meeting on 18 December 2025. The monetary policy statement ranks at the 54th percentile and the answers in the Q&A at the 26th.

The ECB expects inflation to stabilise at two per cent in the medium term, with underlying inflation indicators remaining consistent [1]. The economy remains resilient, supported by low unemployment and solid private sector balance sheets, though growth faces uncertainties from global trade and geopolitical tensions [2]. The ECB will maintain a data-dependent approach to monetary policy, focusing on inflation outlook and risks without pre-committing to a rate path [3].

The main risks include a deterioration in global financial m

## 7. Decisión

**Informe: Llama 3.1 8B Instruct.** Las comprobaciones automáticas empatan y la regla daba Qwen3 en el último desempate. Decide la revisión manual de fidelidad, con un criterio fijado antes de revisar los informes de Llama de las otras cinco ruedas.

- **Formato.** El formato revisado funciona con los dos modelos: las 60 frases citan al menos un fragmento, ninguna cifra es inventada, no hay lenguaje de recomendación y los doce informes cumplen el criterio de aceptación.
- **Por qué no decide el desempate.** El último desempate favorecía a Qwen3 porque ya está en memoria para la postura y el chat. Esa ventaja no existe en el informe: se genera una vez por rueda, fuera de la app, que solo lee `summary.json`.
- **Revisión manual.** Cada una de las 60 frases se comparó con el texto completo de los fragmentos que cita (detalle en `results/informe/revision_manual.csv`). Cuenta como error una cifra o un periodo mal atribuidos, una dirección cambiada o una afirmación que el fragmento citado no respalda. Resultado: 8 frases con error en Qwen3 y 3 en Llama.
  - *Errores que cambian el mensaje:* 2 en cada modelo. Qwen3 presenta la volatilidad de los mercados financieros como un riesgo al alza para la inflación (30 de abril; la declaración dice que la reduciría) y extiende a 2028 la revisión al alza del crecimiento (10 de septiembre). Llama convierte en previsión el impulso que la economía mostraba antes de la guerra (30 de abril) y atribuye al BCE la formulación de una pregunta de un periodista (23 de julio).
  - *Afirmaciones que el fragmento citado no respalda:* 6 en Qwen3 y 1 en Llama. Qwen3 parafrasea y combina información de varios párrafos, de modo que la cita no siempre contiene lo que dice la frase. Llama reproduce casi literalmente el texto citado: al abrir la cita, el usuario encuentra lo mismo que lee, que es la promesa del producto.
- **Coste.** Los informes de Llama son más largos (211 palabras de media frente a 166; unos 85 s de audio frente a 65) y tardan unos 10 s más en generarse, sin efecto en la app. El pipeline necesita un segundo modelo (4,5 GB) para generar informes de ruedas nuevas. Llama 3.1 se distribuye con la Llama 3.1 Community License, que permite el uso comercial.
- **Límites.** La revisión la hizo una sola persona, sin ocultar qué modelo había escrito cada informe. Los errores de Llama vienen de elegir frases fuera de contexto (una pregunta de un periodista, una observación pasada leída como previsión); por eso los informes de la demo se revisan antes de presentarlos, como haría un analista antes de publicar.
- **Informes definitivos.** La sección 8 guarda los informes de Llama a partir de los ya generados, sin volver a llamar al LLM.

## 8. Informes definitivos

Se guardan `stance.csv` y `summary.json` de las seis ruedas con los informes de Llama 3.1 ya generados en la sección 3 (`results/informe/informes.json`). La celda es autónoma: solo necesita la primera celda del notebook (importaciones).

In [2]:
# Informes definitivos con Llama 3.1 8B (ver la decisión): se reutilizan los ya generados, sin volver a llamar al LLM
DEFINITIVO = "mlx-community/Meta-Llama-3.1-8B-Instruct-4bit"
corpus, frases, indice = cargar_historico()
RUEDAS = sorted(indice["date"])[-6:]
generados = {(i["modelo"], i["fecha"]): i
             for i in json.loads((resultados_dir / "informes.json").read_text(encoding="utf-8"))}

for fecha in RUEDAS:
    guardar_informe(preparar_evento(fecha, corpus, frases, indice), generados[(DEFINITIVO, fecha)], DEFINITIVO)
print(f"Guardados en data/events/ ({len(RUEDAS)} ruedas) con {DEFINITIVO}\n")

for fecha in RUEDAS:
    resumen_rueda = json.loads((event_dir(fecha) / "summary.json").read_text(encoding="utf-8"))
    print(f"=== {fecha} | {resumen_rueda['generated_with']['llm']} ===\n\n{resumen_rueda['briefing']}\n")

Guardados en data/events/ (6 ruedas) con mlx-community/Meta-Llama-3.1-8B-Instruct-4bit

=== 2026-02-05 | mlx-community/Meta-Llama-3.1-8B-Instruct-4bit ===

The ECB kept its key interest rates unchanged. The Hawk & Dove stance index rates the press conference as neutral: it ranks at the 42nd percentile of the 50 press conferences held since December 2019, down from the 59th percentile at the previous meeting on 18 December 2025. The monetary policy statement ranks at the 54th percentile and the answers in the Q&A at the 26th.

The ECB expects inflation to stabilise at its two per cent target in the medium term [1]. The economy remains resilient in a challenging global environment, driven by low unemployment, solid private sector balance sheets, and supportive effects of past interest rate cuts [1]. The ECB is determined to ensure that inflation stabilises at its two per cent target in the medium term, following a data-dependent and meeting-by-meeting approach to determining the appropri